In [ ]:
import pandas as pd
import scipy.stats as stats
import itertools
from IPython.display import display

In [ ]:
#Recupération du fichier d'entrainement
df = pd.read_csv("Reproduction_Himmelblau_Final-V2.csv")

N_LIST = [250, 500, 1000, 5000, 10000, 20000]
METRICS = ["MAE_Energy", "MAE_Force"]

results_table = []

for metric in METRICS:
    for n in N_LIST:
        df_sub = df[(df['Training_Labels'] == n) & (df[metric].notnull())]
        strategies = sorted(df_sub['Strategy'].unique())

        data_per_strat = {s: df_sub[df_sub['Strategy'] == s][metric].values for s in strategies}

        if len(strategies) >= 2:
            for s1, s2 in itertools.combinations(strategies, 2):
                d1 = data_per_strat[s1]
                d2 = data_per_strat[s2]

                if len(d1) > 0 and len(d2) > 0:
                    _, p_pair = stats.mannwhitneyu(d1, d2, alternative='two-sided')

                    mean1, mean2 = d1.mean(), d2.mean()

                    is_sig = p_pair < 0.05

                    if is_sig:
                        winner = s1 if mean1 < mean2 else s2
                    else:
                        winner = "Égalité"

                    results_table.append({
                        "Métrique": metric,
                        "N (Exemples)": n,
                        "Stratégie A": s1,
                        "Stratégie B": s2,
                        "Moyenne A": f"{mean1:.4f}",
                        "Moyenne B": f"{mean2:.4f}",
                        "p-value": f"{p_pair:.4e}",
                        "Différence Significative": "Oui" if is_sig else "Non",
                        "Meilleur": winner
                    })

df_stats = pd.DataFrame(results_table)

csv_out = "Analyse_Statistique_Himmelblau_V2.csv"
df_stats.to_csv(csv_out, index=False)
print(f"Analyse terminée et sauvegardée dans '{csv_out}'\n")

display(df_stats)